# **StanceEval2026**
### **Arabic Stance Detection Shared Task Baseline**


This notebook provides a clean and reproducible baseline for the
StanceEval2026 shared task on target-specific Arabic stance detection
using AraBERT-twitter.

---

## **Baseline Configuration**
This baseline is provided as a simple reference implementation.
Participants may adapt and extend any component of the pipeline,
including preprocessing, training strategy, and model selection.

| Setting | Value |
|---|---|
| Model | `aubmindlab/bert-base-arabertv02-twitter` |
| Input format | `(target, text)` |
| Max length | `128` |
| Batch size | `32` |
| Epochs | `20` |
| Learning rate | `2e-5` |
| Best checkpoint | highest dev `Favg2` |
| Labels | `Against`, `Favor`, `None` |

---

## **Features**

- Arabic preprocessing
- Dev evaluation using:
  - `Favg2`
  - `Favg3`
- Submission file generation for:
  - `test_seen.csv`
  - `test_unseen.csv`

---

## **Output Files**

- `submission_seen.csv`
- `submission_unseen.csv`

This baseline is intended to provide participants with a simple
and reproducible starting point for experimentation.

---
## **Expected Data Structure**

The notebook expects the following files inside the `data/` directory:

- `train.csv`
- `dev.csv`
- `test_seen.csv`
- `test_unseen.csv`

## **Requirements**

Recommended environment:

- Python 3.10+
- PyTorch 2.x
- Transformers 4.x

Main packages:
- transformers
- torch
- pandas
- scikit-learn
- tqdm
- sentencepiece

In [ ]:
# ============================================================
# 1) Install Required Packages
# ============================================================
# Skip this cell if packages are already installed
#!pip install -q transformers scikit-learn pandas tqdm sentencepiece

In [ ]:
# ============================================================
# 2) Optional: Mount Google Drive (Colab)
# ============================================================

# Run this cell only if you use Google Drive in Colab

#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# ============================================================
# 3) Imports
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm.auto import tqdm

In [ ]:
# ============================================================
# 4) Configuration
# ============================================================

# =========================
# PATHS
# =========================
# Expected structure:
# project/
# ├── baseline.ipynb
# └── data/
#     ├── train.csv
#     ├── dev.csv
#     ├── test_seen.csv
#     └── test_unseen.csv

BASE_DIR = "."

# If using Google Drive, comment the line above and use this instead:
# BASE_DIR = "/content/drive/MyDrive/StanceEval2026"

DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
DEV_PATH = f"{DATA_DIR}/dev.csv"
TEST_SEEN_PATH = f"{DATA_DIR}/test_seen.csv"
TEST_UNSEEN_PATH = f"{DATA_DIR}/test_unseen.csv"

OUTPUT_DIR = f"{BASE_DIR}/baseline_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# COLUMNS
# =========================
TEXT_COL = "text"
TARGET_COL = "target"
LABEL_COL = "stance"
ID_COL = "id"

# =========================
# MODEL SETTINGS
# =========================
MODEL_DISPLAY_NAME = "AraBERTv0.2-Twitter"
MODEL_HF_NAME = "aubmindlab/bert-base-arabertv02-twitter"

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-5
SEED = 42

LABEL2ID = {
    "Against": 0,
    "Favor": 1,
    "None": 2,
}

ID2LABEL = {v: k for k, v in LABEL2ID.items()}

required_files = [TRAIN_PATH, DEV_PATH, TEST_SEEN_PATH, TEST_UNSEEN_PATH]

for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"File not found: {path}\n"
            "Please make sure all files are inside the data/ folder."
        )

print("All dataset files found.")
print("Model:", MODEL_DISPLAY_NAME)
print("Output directory:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 5) Reproducibility
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# ============================================================
# 6) Arabic Text Preprocessing
# ============================================================

ARABIC_DIACRITICS = re.compile(r"ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ")
NON_ARABIC = re.compile(r"[^\u0600-\u06FF0-9\s]+")
MULTI_SPACE = re.compile(r"\s+")
REPEATED_CHAR = re.compile(r"(.)\1{2,}")

def preprocess_text(text):
    text = str(text)
    text = re.sub(ARABIC_DIACRITICS, "", text)
    text = re.sub(NON_ARABIC, " ", text)
    text = re.sub(REPEATED_CHAR, r"\1\1", text)
    text = re.sub(MULTI_SPACE, " ", text).strip()
    return text

In [ ]:
# ============================================================
# 7) Load Labeled Data (Train / Dev)
# ============================================================

def load_labeled_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)

    needed = [TEXT_COL, TARGET_COL, LABEL_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    for c in needed:
        df[c] = df[c].astype(str).str.strip()

    df = df[
        (df[TEXT_COL] != "") &
        (df[TARGET_COL] != "") &
        (df[LABEL_COL] != "")
    ].copy()

    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)

    unknown_labels = sorted(set(df[LABEL_COL]) - set(LABEL2ID.keys()))
    if unknown_labels:
        raise ValueError(f"Unknown stance labels: {unknown_labels}")

    df["label"] = df[LABEL_COL].map(LABEL2ID).astype(int)
    return df

# ============================================================
# 8) Load Unlabeled Test Data
# ============================================================
def load_unlabeled_test_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)

    needed = [ID_COL, TEXT_COL, TARGET_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")

    for c in needed:
        df[c] = df[c].astype(str).str.strip()

    df = df[
        (df[ID_COL] != "") &
        (df[TEXT_COL] != "") &
        (df[TARGET_COL] != "")
    ].copy()

    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)
    return df


train_df = load_labeled_dataframe(TRAIN_PATH)
dev_df = load_labeled_dataframe(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)

print("\nTrain label counts:")
print(train_df[LABEL_COL].value_counts())

print("\nDev label counts:")
print(dev_df[LABEL_COL].value_counts())

In [ ]:

# ============================================================
# 9) Dataset
# ============================================================
class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoding = self.tokenizer(
            row[TARGET_COL],
            row[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in encoding.items()}

        if not self.is_test:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)

        return item

In [ ]:
# ============================================================
# 10) Evaluation Metrics
# ============================================================

def compute_metrics(y_true, y_pred):
    f_against = f1_score(y_true, y_pred, labels=[0], average="macro", zero_division=0)
    f_favor = f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    f_none = f1_score(y_true, y_pred, labels=[2], average="macro", zero_division=0)

    favg2 = (f_favor + f_against) / 2.0
    favg3 = (f_favor + f_against + f_none) / 3.0
    acc = accuracy_score(y_true, y_pred)

    return {
        "F_favor": f_favor,
        "F_against": f_against,
        "F_none": f_none,
        "Favg2": favg2,
        "Favg3": favg3,
        "Acc": acc,
    }


@torch.no_grad()
def predict_labeled_loader(model, loader):
    model.eval()
    preds = []
    labels = []

    for batch in loader:
        labels.extend(batch["labels"].cpu().numpy().tolist())
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        pred = torch.argmax(outputs.logits, dim=1)

        preds.extend(pred.cpu().numpy().tolist())

    return labels, preds


@torch.no_grad()
def predict_unlabeled_loader(model, loader):
    model.eval()
    preds = []

    for batch in tqdm(loader, desc="Predicting"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        outputs = model(**batch)
        pred = torch.argmax(outputs.logits, dim=1)

        preds.extend(pred.cpu().numpy().tolist())

    return preds


def build_per_target_results(eval_df, pred_ids, model_name):
    df = eval_df.copy()
    df["pred"] = pred_ids

    row = {"Model": model_name}

    for target in sorted(df[TARGET_COL].unique()):
        sub = df[df[TARGET_COL] == target]
        m = compute_metrics(sub["label"], sub["pred"])
        safe_target = target.replace(" ", "_").replace("/", "_")

        row[f"{safe_target}_Favg2"] = m["Favg2"] * 100
        row[f"{safe_target}_Favg3"] = m["Favg3"] * 100

    overall = compute_metrics(df["label"], df["pred"])

    row["F_favor"] = overall["F_favor"] * 100
    row["F_against"] = overall["F_against"] * 100
    row["F_none"] = overall["F_none"] * 100
    row["Overall_Favg2"] = overall["Favg2"] * 100
    row["Overall_Favg3"] = overall["Favg3"] * 100
    row["Acc"] = overall["Acc"] * 100

    return pd.DataFrame([row]).round(2), df

In [ ]:
# ============================================================
# 11) Tokenizer + DataLoaders
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_NAME)

train_ds = StanceDataset(train_df, tokenizer)
dev_ds = StanceDataset(dev_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# 12) Load Model
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_HF_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
).to(DEVICE)

optimizer = AdamW(model.parameters(), lr=LR)

In [ ]:
# ============================================================
# 13) Training + Dev Evaluation
# ============================================================
model_base_dir = f"{OUTPUT_DIR}/{MODEL_DISPLAY_NAME}"
best_favg2_dir = f"{model_base_dir}/best_favg2"

os.makedirs(best_favg2_dir, exist_ok=True)

history = []
best_dev_favg2 = -1.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0

    pbar = tqdm(train_loader, desc=f"{MODEL_DISPLAY_NAME} | Epoch {epoch+1}/{EPOCHS}")

    for batch in pbar:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = total_loss / max(len(train_loader), 1)

    dev_labels, dev_preds = predict_labeled_loader(model, dev_loader)
    dev_metrics = compute_metrics(dev_labels, dev_preds)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "dev_F_favor": dev_metrics["F_favor"] * 100,
        "dev_F_against": dev_metrics["F_against"] * 100,
        "dev_F_none": dev_metrics["F_none"] * 100,
        "dev_Favg2": dev_metrics["Favg2"] * 100,
        "dev_Favg3": dev_metrics["Favg3"] * 100,
        "dev_Acc": dev_metrics["Acc"] * 100,
    })

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"dev_Favg2={dev_metrics['Favg2']*100:.2f} | "
        f"dev_Favg3={dev_metrics['Favg3']*100:.2f} | "
        f"dev_acc={dev_metrics['Acc']*100:.2f}"
    )

    if dev_metrics["Favg2"] > best_dev_favg2:
        best_dev_favg2 = dev_metrics["Favg2"]
        model.save_pretrained(best_favg2_dir)
        tokenizer.save_pretrained(best_favg2_dir)
        print("Saved BEST_FAVG2 checkpoint.")

history_df = pd.DataFrame(history)
history_path = f"{model_base_dir}/training_history.csv"
history_df.to_csv(history_path, index=False)

print("\nTraining history saved to:", history_path)
display(history_df)

In [ ]:
# ============================================================
# 14) Final Dev Evaluation
# ============================================================

best_tokenizer = AutoTokenizer.from_pretrained(best_favg2_dir)
best_model = AutoModelForSequenceClassification.from_pretrained(best_favg2_dir).to(DEVICE)

dev_ds_best = StanceDataset(dev_df, best_tokenizer)
dev_loader_best = DataLoader(dev_ds_best, batch_size=BATCH_SIZE, shuffle=False)

dev_labels, dev_preds = predict_labeled_loader(best_model, dev_loader_best)
dev_metrics = compute_metrics(dev_labels, dev_preds)

print("\nFINAL DEV RESULTS USING BEST_FAVG2 CHECKPOINT")
print(f"F_favor:   {dev_metrics['F_favor']*100:.2f}")
print(f"F_against: {dev_metrics['F_against']*100:.2f}")
print(f"F_none:    {dev_metrics['F_none']*100:.2f}")
print(f"Favg2:     {dev_metrics['Favg2']*100:.2f}")
print(f"Favg3:     {dev_metrics['Favg3']*100:.2f}")
print(f"Accuracy:  {dev_metrics['Acc']*100:.2f}")

print("\nDEV CLASSIFICATION REPORT")
print(
    classification_report(
        dev_labels,
        dev_preds,
        labels=[0, 1, 2],
        target_names=["Against", "Favor", "None"],
        digits=4,
        zero_division=0
    )
)

dev_table, dev_predictions_df = build_per_target_results(
    dev_df,
    dev_preds,
    f"{MODEL_DISPLAY_NAME}_best_favg2"
)

print("\nDEV RESULTS PER TARGET USING BEST_FAVG2 CHECKPOINT")
display(dev_table)

dev_table_path = f"{model_base_dir}/dev_results_per_target_best_favg2.csv"
dev_table.to_csv(dev_table_path, index=False)

dev_predictions_df["pred_id"] = dev_predictions_df["pred"]
dev_predictions_df["pred_stance"] = [ID2LABEL[p] for p in dev_predictions_df["pred"]]

dev_pred_path = f"{model_base_dir}/dev_predictions_best_favg2.csv"
dev_predictions_df.to_csv(dev_pred_path, index=False)

print("\nDev predictions saved to:", dev_pred_path)
print("Dev per-target results saved to:", dev_table_path)

In [ ]:
# ============================================================
# 15) Submission Generation: Predict on Unlabeled Test Files
# ============================================================

def generate_submission(test_path, output_filename):
    test_df = load_unlabeled_test_dataframe(test_path)

    test_ds = StanceDataset(test_df, best_tokenizer, is_test=True)

    test_loader = DataLoader(
        test_ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    pred_ids = predict_unlabeled_loader(best_model, test_loader)

    pred_labels = [ID2LABEL[p] for p in pred_ids]

    submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        LABEL_COL: pred_labels
    })

    output_path = f"{model_base_dir}/{output_filename}"

    submission.to_csv(output_path, index=False)

    print("Saved submission to:", output_path)

    return submission

In [ ]:
# ============================================================
# 16) Generate Seen and Unseen Submissions
# ============================================================

submission_seen = generate_submission(
    TEST_SEEN_PATH,
    "submission_seen.csv"
)

submission_unseen = generate_submission(
    TEST_UNSEEN_PATH,
    "submission_unseen.csv"
)

## **Notes**

- This baseline is intended as a simple and reproducible reference implementation for the shared task.
- Participants are encouraged to experiment with different preprocessing methods, training strategies, and model architectures.
- Performance may vary depending on hardware, random initialization, and software environment.